# SAM 2


## Imports


In [ ]:
sys.path.insert(0, "/home/vteam5/sam2")
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


## Path Configuration


In [ ]:
sam2_checkpoint = model_checkpoints_dir / "sam2_hiera_large.pt"
SAM2_URL = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt"

if not sam2_checkpoint.exists():
    print("Downloading SAM 2 checkpoint. This only runs when the file is missing.")
    !wget -q --show-progress -O "{sam2_checkpoint}" "{SAM2_URL}"
    if not sam2_checkpoint.exists() or sam2_checkpoint.stat().st_size < 1e8:
        raise RuntimeError("SAM 2 checkpoint download failed. Re-run this cell or check connectivity.")
else:
    print("SAM 2 checkpoint already exists.")

print(f"Project root: {project_root}")
print(f"Ensemble results folder: {ensemble_results_dir}")


SAM 2 checkpoint already exists.
Project root: /home/vteam5/multispectral_pedestrian_ensemble
Ensemble results folder: /home/vteam5/multispectral_pedestrian_ensemble/results_ensemble_v2


## SAM 2 Setup


In [ ]:
SAM2_CONFIG = "configs/sam2.1/sam2.1_hiera_l.yaml"
print("Loading SAM 2 Hiera-Large.")
sam2_base = build_sam2(config_file=SAM2_CONFIG, ckpt_path=str(sam2_checkpoint), device=compute_device)
sam2_pred = SAM2ImagePredictor(sam2_base)
print("SAM 2 loaded.")


Loading SAM 2 Hiera-Large.
SAM 2 loaded.


In [ ]:
def choose_best_sam_mask(raw_masks, raw_iou, box_index):
    """Pick the highest-scoring SAM mask for one YOLO box."""
    masks = np.asarray(raw_masks)
    scores = np.asarray(raw_iou)

    if masks.ndim == 4:
        candidate_masks = masks[box_index]
        candidate_scores = scores[box_index].reshape(-1)
    elif masks.ndim == 3:
        candidate_masks = masks
        candidate_scores = scores.reshape(-1)
    else:
        candidate_masks = masks.reshape(1, *masks.shape[-2:])
        candidate_scores = scores.reshape(-1)

    best_index = int(np.argmax(candidate_scores)) if len(candidate_scores) else 0
    return candidate_masks[best_index].astype(bool), float(candidate_scores[best_index])


def mask_to_box(mask):
    rows, cols = np.where(mask)
    if len(rows) == 0:
        return None
    return [float(cols.min()), float(rows.min()), float(cols.max()), float(rows.max())]


In [ ]:
def validate_sam_mask(mask, yolo_box, min_area_ratio=0.05, max_area_ratio=1.25, min_box_iou=0.30):
    """Reject only masks that are clearly empty, too large, too tiny, or far from the YOLO box."""
    mask_box = mask_to_box(mask)
    if mask_box is None:
        return False, "empty_mask", None

    yolo_area = max(1.0, (yolo_box[2] - yolo_box[0]) * (yolo_box[3] - yolo_box[1]))
    mask_area = float(mask.sum())
    area_ratio = mask_area / yolo_area
    overlap = box_iou(mask_box, yolo_box)

    if area_ratio < min_area_ratio:
        return False, "mask_too_small", mask_box
    if area_ratio > max_area_ratio:
        return False, "mask_too_large", mask_box
    if overlap < min_box_iou:
        return False, "mask_far_from_box", mask_box
    return True, "kept", mask_box


## Veto Threshold Logic

In [ ]:
# SAM uses visible RGB because that is closer to the images it was trained on.
# YOLO still uses the fused image because the checkpoint was trained on fused inputs.
SAM_MASK_MIN_AREA_RATIO = 0.05
SAM_MASK_MAX_AREA_RATIO = 1.25
SAM_MASK_MIN_BOX_IOU = 0.30
cascade_conf = selected_conf_for_cascade if "selected_conf_for_cascade" in globals() else conf_default

print(f"Cascade confidence: {cascade_conf:.2f}")
print(f"Minimum mask/box area ratio: {SAM_MASK_MIN_AREA_RATIO:.2f}")
print(f"Maximum mask/box area ratio: {SAM_MASK_MAX_AREA_RATIO:.2f}")
print(f"Minimum mask-box IoU: {SAM_MASK_MIN_BOX_IOU:.2f}")


Cascade confidence: 0.30
Minimum mask/box area ratio: 0.05
Maximum mask/box area ratio: 1.25
Minimum mask-box IoU: 0.30


In [ ]:
def run_sam2_refinement(visible_rgb: np.ndarray, boxes_xyxy: np.ndarray):
    """Run SAM 2 on visible RGB and keep one checked mask per YOLO box."""
    if len(boxes_xyxy) == 0:
        return [], [], [], []

    kept_masks = []
    kept_scores = []
    kept_boxes = []
    veto_reasons = []

    with torch.inference_mode():
        sam2_pred.set_image(visible_rgb)
        raw_masks, raw_iou, _ = sam2_pred.predict(
            point_coords=None,
            point_labels=None,
            box=boxes_xyxy,
            multimask_output=True,
        )

    for box_index, yolo_box in enumerate(boxes_xyxy):
        mask, score = choose_best_sam_mask(raw_masks, raw_iou, box_index)
        keep, reason, mask_box = validate_sam_mask(
            mask,
            yolo_box,
            min_area_ratio=SAM_MASK_MIN_AREA_RATIO,
            max_area_ratio=SAM_MASK_MAX_AREA_RATIO,
            min_box_iou=SAM_MASK_MIN_BOX_IOU,
        )
        if keep:
            kept_masks.append(mask)
            kept_scores.append(score)
            kept_boxes.append(mask_box)
        else:
            veto_reasons.append(reason)

    return kept_masks, kept_scores, kept_boxes, veto_reasons


In [ ]:
def resize_visible_for_sam(vis_path: Path, res: int = TARGET_RES) -> np.ndarray:
    """Load the original visible image and resize it to match YOLO boxes."""
    visible_bgr = cv2.imread(str(vis_path))
    if visible_bgr is None:
        raise FileNotFoundError(f"Visible image not found: {vis_path}")
    visible_bgr = cv2.resize(visible_bgr, (res, res), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(visible_bgr, cv2.COLOR_BGR2RGB)


In [ ]:
veto_settings = {
    "cascade_confidence": cascade_conf,
    "min_area_ratio": SAM_MASK_MIN_AREA_RATIO,
    "max_area_ratio": SAM_MASK_MAX_AREA_RATIO,
    "min_box_iou": SAM_MASK_MIN_BOX_IOU,
    "sam_input": "original_visible_rgb_resized_to_1024",
}

veto_settings_path = ensemble_results_dir / "veto_settings.json"
with open(veto_settings_path, "w") as f:
    json.dump(veto_settings, f, indent=2)
print(f"Saved veto settings: {veto_settings_path}")


Saved veto settings: /home/vteam5/multispectral_pedestrian_ensemble/results_ensemble_v2/veto_settings.json
